In [2]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

In [3]:
import subprocess
print("Full Dataset Size:\n")
result = subprocess.run(['du', '-sb', '/kaggle/input/competitions/diabetic-retinopathy-detection'],capture_output=True, text=True)
total_bytes = int(result.stdout.split()[0])
size_gb  = total_bytes / 1_000_000_000      # dec
size_gib = total_bytes / (1024 ** 3)        # binary
print(f"Decimal (GB): {size_gb:.2f} GB")
print(f"Binary  (GiB): {size_gib:.2f} GiB")

Full Dataset Size:

Decimal (GB): 88.29 GB
Binary  (GiB): 82.23 GiB


In [4]:
import os
input_dir = '/kaggle/input/competitions/diabetic-retinopathy-detection'
print("Dataset files directory:\n")
for item in sorted(os.listdir(input_dir)):
    print(f"  {item}")

Dataset files directory:

  sample.zip
  sampleSubmission.csv.zip
  test.zip.001
  test.zip.002
  test.zip.003
  test.zip.004
  test.zip.005
  test.zip.006
  test.zip.007
  train.zip.001
  train.zip.002
  train.zip.003
  train.zip.004
  train.zip.005
  trainLabels.csv.zip


In [ ]:
import os
import cv2
import math
import shutil
import subprocess
import glob
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

possible_zips = glob.glob("/kaggle/input/**/*train.zip.001", recursive=True)

possible_csvs = glob.glob("/kaggle/input/**/*trainLabels*.csv", recursive=True)
possible_csv_zips = glob.glob("/kaggle/input/**/*trainLabels*.csv.zip", recursive=True)

if len(possible_zips) == 0:
    raise FileNotFoundError("train.zip.001 not found.")

FIRST_PART = possible_zips[0]

if len(possible_csvs) > 0:
    CSV_PATH = possible_csvs[0]
elif len(possible_csv_zips) > 0:
    CSV_ZIP_PATH = possible_csv_zips[0]
    CSV_EXTRACT_DIR = "/kaggle/working/labels"
    Path(CSV_EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

    print("Extracting label CSV from:", CSV_ZIP_PATH)

    shutil.unpack_archive(
        CSV_ZIP_PATH,
        CSV_EXTRACT_DIR,
        "zip"
    )

    extracted_csvs = glob.glob(CSV_EXTRACT_DIR + "/**/*.csv", recursive=True)

    if len(extracted_csvs) == 0:
        raise FileNotFoundError("Could not extract trainLabels.csv from zip.")

    CSV_PATH = extracted_csvs[0]

else:
    raise FileNotFoundError("trainLabels.csv or trainLabels.csv.zip not found.")

print("Using ZIP:", FIRST_PART)
print("Using CSV:", CSV_PATH)

# i splited the dataset considering patient id, diseaselevel and left and right eye, 
# for the distribution of the dataset is preserved in all splits 
# and there is no data leakage between splits. 
TEMP_DIR = "/kaggle/working/temp_extract"
OUTPUT_DIR = "/kaggle/working/SplitDATA"
LISTING_TXT = "/kaggle/working/train_listing.txt"



TARGET_SIZE = (384, 384)
QUALITY = 85
CHUNK_SIZE = 3000
SEED = 42

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

project_path = Path(OUTPUT_DIR)

folders = [
    "data/train/0", "data/train/1", "data/train/2", "data/train/3", "data/train/4",
    "data/val/0", "data/val/1", "data/val/2", "data/val/3", "data/val/4",
    "data/test/0", "data/test/1", "data/test/2", "data/test/3", "data/test/4",
    "checkpoints",
    "logs"
]

for folder in folders:
    (project_path / folder).mkdir(parents=True, exist_ok=True)
    print(f"✓ Created: {folder}")

def run_cmd(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode, result.stdout, result.stderr

def clear_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def show_disk():
    code, out, err = run_cmd(["df", "-h", "/kaggle/working"])
    print(out if out else err)

def count_images(folder):
    total = 0
    for _, _, files in os.walk(folder):
        total += sum(f.lower().endswith((".jpeg", ".jpg", ".png")) for f in files)
    return total

def crop_black_border(img, threshold=10):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > threshold

    if mask.sum() == 0:
        return img

    coords = np.argwhere(mask)
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    return img[y0:y1, x0:x1]

def preprocess_image(img):
    img = crop_black_border(img)
    img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    return img

df = pd.read_csv(CSV_PATH)
print("\nLoaded labels:", len(df))

df["patient_id"] = df["image"].apply(lambda x: x.split("_")[0])

patient_df = df.groupby("patient_id")["level"].max().reset_index()

train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.40,
    random_state=SEED,
    stratify=patient_df["level"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_patients["level"]
)

train_df = df[df["patient_id"].isin(train_patients["patient_id"])].reset_index(drop=True)
val_df = df[df["patient_id"].isin(val_patients["patient_id"])].reset_index(drop=True)
test_df = df[df["patient_id"].isin(test_patients["patient_id"])].reset_index(drop=True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

total_images = len(split_df)

print("\n========== FINAL SPLITS ==========")
print(f"Train images: {len(train_df)} ({100 * len(train_df) / total_images:.2f}%)")
print(f"Val images:   {len(val_df)} ({100 * len(val_df) / total_images:.2f}%)")
print(f"Test images:  {len(test_df)} ({100 * len(test_df) / total_images:.2f}%)")

print("\n========== PATIENT COUNTS ==========")
print("Train patients:", train_df["patient_id"].nunique())
print("Val patients:", val_df["patient_id"].nunique())
print("Test patients:", test_df["patient_id"].nunique())

print("\n========== CLASS DISTRIBUTION ==========")
print("\nTrain:")
print(train_df["level"].value_counts(normalize=True).sort_index())

print("\nVal:")
print(val_df["level"].value_counts(normalize=True).sort_index())

print("\nTest:")
print(test_df["level"].value_counts(normalize=True).sort_index())

train_df.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test.csv"), index=False)

print("\nSaved split CSV files")

image_to_split = dict(zip(split_df["image"], split_df["split"]))
image_to_label = dict(zip(split_df["image"], split_df["level"]))

print("\nReading archive listing...")

code, out, err = run_cmd(["7z", "l", FIRST_PART])

if code != 0:
    print("Cannot read archive")
    print(err)
    raise SystemExit

with open(LISTING_TXT, "w") as f:
    f.write(out)

image_files = []

for line in out.splitlines():
    line = line.strip()

    if not line:
        continue

    if " train/" in " " + line and line.lower().endswith((".jpeg", ".jpg", ".png")):
        filename = line.split()[-1]

        if filename.startswith("train/"):
            base_name = os.path.splitext(os.path.basename(filename))[0]

            if base_name in image_to_split:
                image_files.append(filename)

seen = set()
image_files = [x for x in image_files if not (x in seen or seen.add(x))]

print(f"\nTotal labelled image files found: {len(image_files)}")
print("Sample files:", image_files[:5])

if len(image_files) == 0:
    raise SystemExit("No images found in archive.")

print("\nRunning extraction test...")

clear_dir(TEMP_DIR)

test_files = image_files[:2]

code, out, err = run_cmd([
    "7z",
    "x",
    FIRST_PART,
    f"-o{TEMP_DIR}",
    "-y",
    *test_files
])

print("7z return code:", code)

test_extracted = []

for root, dirs, files in os.walk(TEMP_DIR):
    for f in files:
        if f.lower().endswith((".jpeg", ".jpg", ".png")):
            test_extracted.append(os.path.join(root, f))

print("Extracted test files:", len(test_extracted))

if len(test_extracted) == 0:
    raise SystemExit("Extraction test failed.")

clear_dir(TEMP_DIR)
num_chunks = math.ceil(len(image_files) / CHUNK_SIZE)

print("\nStarting chunked processing...")
print("Chunk size:", CHUNK_SIZE)
print("Total chunks:", num_chunks)

show_disk()

total_processed = 0
total_failed = 0

for chunk_idx in range(num_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(image_files))

    chunk_files = image_files[start:end]

    print(f"\nChunk {chunk_idx + 1}/{num_chunks}")
    print(f"Files {start} -> {end - 1}")

    clear_dir(TEMP_DIR)

    cmd = [
        "7z",
        "x",
        FIRST_PART,
        f"-o{TEMP_DIR}",
        "-y"
    ] + chunk_files

    code, out, err = run_cmd(cmd)

    print("7z return code:", code)

    if code != 0:
        print("Extraction failed!")
        print(err[:1000])
        total_failed += len(chunk_files)
        continue

    extracted_paths = []

    for root, dirs, files in os.walk(TEMP_DIR):
        for f in files:
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                extracted_paths.append(os.path.join(root, f))

    print("Extracted files:", len(extracted_paths))

    processed_this_chunk = 0
    failed_this_chunk = 0

    for img_path in tqdm(extracted_paths, desc="Processing"):

        try:
            img_name = os.path.splitext(os.path.basename(img_path))[0]

            if img_name not in image_to_split:
                continue

            split = image_to_split[img_name]
            label = int(image_to_label[img_name])

            save_dir = os.path.join(
                OUTPUT_DIR,
                "data",
                split,
                str(label)
            )

            out_path = os.path.join(save_dir, img_name + ".jpg")

            if os.path.exists(out_path):
                continue

            img = cv2.imread(img_path)

            if img is None:
                failed_this_chunk += 1
                total_failed += 1
                continue

            img = preprocess_image(img)

            ok = cv2.imwrite(
                out_path,
                img,
                [cv2.IMWRITE_JPEG_QUALITY, QUALITY]
            )

            if ok:
                processed_this_chunk += 1
                total_processed += 1
            else:
                failed_this_chunk += 1
                total_failed += 1

        except Exception:
            failed_this_chunk += 1
            total_failed += 1

    print("Processed:", processed_this_chunk)
    print("Failed:", failed_this_chunk)

    print("\nCurrent dataset counts:")
    print("Train:", count_images(os.path.join(OUTPUT_DIR, "data/train")))
    print("Val:", count_images(os.path.join(OUTPUT_DIR, "data/val")))
    print("Test:", count_images(os.path.join(OUTPUT_DIR, "data/test")))

    clear_dir(TEMP_DIR)

    print("\nDisk after cleanup:")
    show_disk()

print("\nProcessing completed!")

print("Total processed:", total_processed)
print("Total failed:", total_failed)

print("\nFinal dataset counts:")
print("Train:", count_images(os.path.join(OUTPUT_DIR, "data/train")))
print("Val:", count_images(os.path.join(OUTPUT_DIR, "data/val")))
print("Test:", count_images(os.path.join(OUTPUT_DIR, "data/test")))

print("\nDataset location:")
print(OUTPUT_DIR)

print("\nDataset size:")
os.system(f'du -sh "{OUTPUT_DIR}"')

ZIP_BASE = "/kaggle/working/SplitDATA"

print("\nCreating zip archive...")

shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

print("ZIP created:")
print(ZIP_BASE + ".zip")

Extracting label CSV from: /kaggle/input/competitions/diabetic-retinopathy-detection/trainLabels.csv.zip
Using ZIP: /kaggle/input/competitions/diabetic-retinopathy-detection/train.zip.001
Using CSV: /kaggle/working/labels/trainLabels.csv
✓ Created: data/train/0
✓ Created: data/train/1
✓ Created: data/train/2
✓ Created: data/train/3
✓ Created: data/train/4
✓ Created: data/val/0
✓ Created: data/val/1
✓ Created: data/val/2
✓ Created: data/val/3
✓ Created: data/val/4
✓ Created: data/test/0
✓ Created: data/test/1
✓ Created: data/test/2
✓ Created: data/test/3
✓ Created: data/test/4
✓ Created: checkpoints
✓ Created: logs

Loaded labels: 35126

========== FINAL SPLITS ==========
Train images: 21074 (60.00%)
Val images:   7026 (20.00%)
Test images:  7026 (20.00%)

========== PATIENT COUNTS ==========
Train patients: 10537
Val patients: 3513
Test patients: 3513

========== CLASS DISTRIBUTION ==========

Train:
level
0    0.735076
1    0.069042
2    0.151087
3    0.024817
4    0.019977
Name: prop

Processing: 100%|██████████| 3000/3000 [08:51<00:00,  5.64it/s]


Processed: 3000
Failed: 0

Current dataset counts:
Train: 1866
Val: 552
Test: 582

Disk after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   57M   20G   1% /kaggle/working


Chunk 2/12
Files 3000 -> 5999
7z return code: 0
Extracted files: 3000


Processing: 100%|██████████| 3000/3000 [08:43<00:00,  5.73it/s]


Processed: 3000
Failed: 0

Current dataset counts:
Train: 3698
Val: 1160
Test: 1142

Disk after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  110M   20G   1% /kaggle/working


Chunk 3/12
Files 6000 -> 8999
7z return code: 0
Extracted files: 3000


...

Processed: 3000
Failed: 0

Current dataset counts:
Train: 18054
Val: 5928
Test: 6018

Disk after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  533M   19G   3% /kaggle/working


Chunk 11/12
Files 30000 -> 32999
7z return code: 0
Extracted files: 3000


Processing: 100%|██████████| 3000/3000 [08:43<00:00,  5.73it/s]


Processed: 3000
Failed: 0

Current dataset counts:
Train: 19812
Val: 6564
Test: 6624

Disk after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  585M   19G   3% /kaggle/working


Chunk 12/12
Files 33000 -> 35125
7z return code: 0
Extracted files: 2126


Processing: 100%|██████████| 2126/2126 [06:17<00:00,  5.64it/s]


Processed: 2126
Failed: 0

Current dataset counts:
Train: 21074
Val: 7026
Test: 7026

Disk after cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  623M   19G   4% /kaggle/working


Processing completed!
Total processed: 35126
Total failed: 0

Final dataset counts:
Train: 21074
Val: 7026
Test: 7026

Dataset location:
/kaggle/working/SplitDATA

Dataset size:
620M	/kaggle/working/SplitDATA

Creating zip archive...
ZIP created:
/kaggle/working/SplitDATA.zip


In [7]:
print("\nProcessed Dataset Size:\n")

result = subprocess.run(
    ['du', '-sb', OUTPUT_DIR],
    capture_output=True,
    text=True
)

processed_bytes = int(result.stdout.split()[0])

processed_gb = processed_bytes / 1_000_000_000
processed_gib = processed_bytes / (1024 ** 3)

print(f"Decimal (GB): {processed_gb:.2f} GB")
print(f"Binary  (GiB): {processed_gib:.2f} GiB")


Processed Dataset Size:

Decimal (GB): 0.58 GB
Binary  (GiB): 0.54 GiB


In [8]:
# # Clear output folder
# import os

# def remove_folder_contents(folder):
#     for the_file in os.listdir(folder):
#         file_path = os.path.join(folder, the_file)
#         try:
#             if os.path.isfile(file_path):
#                 os.unlink(file_path)
#             elif os.path.isdir(file_path):
#                 remove_folder_contents(file_path)
#                 os.rmdir(file_path)
#         except Exception as e:
#             print(e)

# folder_path = '/kaggle/working/'